# The God Paradox
## Does Perfect Timing Beat Dollar-Cost Averaging?

---

### Background

Nick Maggiulli (2019) showed that a God-like investor who buys *every* market bottom
still **loses to DCA ~70% of rolling 40-year windows** when idle cash earns 0%.

This notebook replicates that baseline and extends it with three God strategies,
with real T-bill yield on idle cash.

| Parameter | Value |
|-----------|-------|
| Data | Shiller Real Total Return S&P 500 (ie_data.xls, column J) |
| Monthly contribution | $100 (DCA and God alike) |
| Dividends | Embedded in real_tr — no separate DRIP applied |
| God's idle cash yield (baseline) | 0% — exact Maggiulli (2019) assumption |
| God's idle cash yield (extended) | Real 1-yr T-bill rate (Fisher equation, 12-month trailing CPI from ie_data.xls) |
| Bottom detection | Global: computed once on full 1920–2026 series, filtered per window |
| Date range | Jan 1920 – May 2026 |

### Strategies Tested

| Strategy | Cash yield | Rule |
|----------|-----------|------|
| **DCA** | — | $100/month, fully invested |
| **God Maggiulli (0%)** | 0% | Global inter-ATH trough — Maggiulli 2019 replication |
| **God Maggiulli (real T-bill)** | Real T-bill | Same buy schedule, cash earns real T-bill yield |
| **God 5-Year** | Real T-bill | Once per 5-year period at the period low |
| **God 10-Year** | Real T-bill | Once per decade at the decade low |

### Two Experiments

| # | Experiment | Scope |
|---|------------|-------|
| **A** | Full run | Jan 1920 – May 2026 |
| **B** | Exhaustive rolling windows | All valid 40-year monthly-start windows |

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))

from god_paradox import (
    load_shiller_real_tr, load_bond_yields, load_cpi_series,
    maggiulli_global_bottoms, run_all_strategies,
    CHARTS_DIR,
)
from rolling_windows import (
    run_all_windows_all_strategies,
    print_summary,
    plot_windows_comparison,
)
from charts import plot_buy_timeline, plot_comparison_growth, plot_final_bars

from IPython.display import Image, display as ipy_display
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

print("Loading data…")
prices      = load_shiller_real_tr()
bond_yields = load_bond_yields(prices)
cpi         = load_cpi_series()
print(f"  {len(prices)} monthly observations  "
      f"({prices.index[0].strftime('%b %Y')} – {prices.index[-1].strftime('%b %Y')})")
print(f"  Real TR range: {prices.iloc[0]:.2f} → {prices.iloc[-1]:,.2f}")

---
## Scenario A — Full Run (Jan 1920 – May 2026)

All strategies run on the Shiller Real Total Return index.

**Baseline** (0% cash, no T-bill): replicates Maggiulli's original assumption — validates our implementation.  
**Extended** (real T-bill cash): our contribution — shows how earning real T-bill yield on idle cash changes the outcome.

Global bottom schedule: `maggiulli_global_bottoms()` runs once on the full 1920–2026 series,
faithfully porting Maggiulli's `create_full_dd()` two-pass algorithm from the original R code.

In [ ]:
result = run_all_strategies(prices, bond_yields, cpi)

### When Does God Buy?

Three stacked panels — one per timing strategy. Each panel overlays:
- **Gray line**: S&P 500 price (log scale)
- **Colored stems**: cash deployed at each buy date (stem height = dollar amount; color matches strategy)

Labels show the cash amount for strategies with ≤ 25 buys.

In [ ]:
plot_buy_timeline(result)
ipy_display(Image(str(CHARTS_DIR / 'buy_timeline.png')))

### Portfolio Growth — All Four God Strategies vs DCA

In [ ]:
plot_comparison_growth(result)
ipy_display(Image(str(CHARTS_DIR / 'comparison_growth.png')))

### Final Portfolio Values — 1920–2026

In [ ]:
plot_final_bars(result)
ipy_display(Image(str(CHARTS_DIR / 'final_bars.png')))

In [ ]:
rows = [
    {"Strategy": "DCA",
     "Final Value": f'${result["final_dca"]:,.0f}',
     "vs DCA": "—",
     "# Buys": "every month",
     "Cash yield": "—"},
    {"Strategy": "God Maggiulli (0% cash) — Maggiulli 2019 replication",
     "Final Value": f'${result["final_mag_base"]:,.0f}',
     "vs DCA": f'{result["adv_mag_base"]:+.1f}%',
     "# Buys": len(result['buys_mag_base']),
     "Cash yield": "0%"},
    {"Strategy": "God Maggiulli (real T-bill) — Fisher-corrected",
     "Final Value": f'${result["final_mag"]:,.0f}',
     "vs DCA": f'{result["adv_mag"]:+.1f}%',
     "# Buys": len(result['buys_mag']),
     "Cash yield": "Real T-bill"},
    {"Strategy": "God 5-Year (real T-bill)",
     "Final Value": f'${result["final_5yr"]:,.0f}',
     "vs DCA": f'{result["adv_5yr"]:+.1f}%',
     "# Buys": len(result['buys_5yr']),
     "Cash yield": "Real T-bill"},
    {"Strategy": "God 10-Year (real T-bill)",
     "Final Value": f'${result["final_10yr"]:,.0f}',
     "vs DCA": f'{result["adv_10yr"]:+.1f}%',
     "# Buys": len(result['buys_10yr']),
     "Cash yield": "Real T-bill"},
]
ipy_display(pd.DataFrame(rows).set_index("Strategy"))
# Note: Scenario A starts Jan 1920, so the 10-Year strategy has 11 start-year-anchored blocks.

---
## Scenario B — Exhaustive Rolling Windows

Every valid monthly-start 40-year window in the 1920–2026 dataset is tested.

**Global bottom schedule**: `maggiulli_global_bottoms()` is run once on the full series.
Each window filters the result to dates within its range — faithful replication of
Maggiulli's original R code (`create_full_dd` + `filter(date >= start, date <= end)`).

Four strategies compared:
- **Baseline (0% cash)**: Maggiulli timing, 0% cash yield → replication of Maggiulli (2019)
- **Maggiulli (real T-bill)**: same timing, real T-bill yield on cash
- **5-Year (real T-bill)**: window-relative 5-year period minimum, real T-bill cash
- **10-Year (real T-bill)**: window-relative 10-year period minimum, real T-bill cash

In [ ]:
print("Running exhaustive rolling windows for all four strategies…")
print("(Global bottoms computed once; ~4 × windows simulations — takes 2–4 minutes)")
w_base, w_mag, w_5yr, w_10yr = run_all_windows_all_strategies(prices, bond_yields, cpi)
print("Done.")

In [ ]:
print_summary(w_base, "BASELINE (0% CASH — MAGGIULLI 2019 REPLICATION)")
print_summary(w_mag,  "GOD MAGGIULLI (REAL T-BILL CASH)")
print_summary(w_5yr,  "GOD 5-YEAR (REAL T-BILL)")
print_summary(w_10yr, "GOD 10-YEAR (REAL T-BILL)")

### Rolling Window Comparison Chart

Four panels:
1. **Scatter**: God's advantage (%) vs window start year — all 4 strategies overlaid
2. **Histogram**: distribution of God's advantage — all 4 overlaid (dashed lines = medians)
3. **Grouped bars**: God win rate by starting decade for each strategy
4. **Scorecard**: summary statistics table (std dev uses sample std, ddof=1)

Baseline win rate should be near Maggiulli's published ~30% God win rate, validating the replication.

In [ ]:
plot_windows_comparison(w_base, w_mag, w_5yr, w_10yr)
ipy_display(Image(str(CHARTS_DIR / 'rolling_comparison.png')))

In [ ]:
def _row(name, results):
    adv = [r.god_advantage for r in results]
    gw  = sum(r.god_wins for r in results)
    return {
        "Strategy":      name,
        "Windows":       len(results),
        "God Win Rate":  f'{gw / len(results) * 100:.0f}%  ({gw}/{len(results)})',
        "Median Adv":    f'{np.median(adv):+.1f}%',
        "Mean Adv":      f'{np.mean(adv):+.1f}%',
        "Std Dev":       f'{np.std(adv, ddof=1):.1f}%',
        "Best (God)":    f'{max(adv):+.1f}%',
        "Worst (DCA)":   f'{min(adv):+.1f}%',
    }

ipy_display(pd.DataFrame([
    _row("Baseline (0% cash — Maggiulli 2019)", w_base),
    _row("Maggiulli (real T-bill cash)",        w_mag),
    _row("5-Year (real T-bill)",                w_5yr),
    _row("10-Year (real T-bill)",               w_10yr),
]).set_index("Strategy"))

---
## Key Findings

### Scenario A — Full Run (Jan 1920 – May 2026)

| Strategy | Final Real Value | vs DCA | # Buys | Cash yield |
|----------|-----------------|--------|--------|-----------:|
| **DCA** | $35,426,472 | — | every month | — |
| **God Mag (0%)** | $37,980,987 | **+7.2%** | 104 | 0% |
| **God Mag (real T-bill)** | $39,989,271 | **+12.9%** | 104 | Real T-bill |
| **God 5-Year** | $46,194,126 | **+30.4%** | 22 | Real T-bill |
| **God 10-Year** | $52,052,778 | **+46.9%** | 11 | Real T-bill |

*Total contributed by each strategy: $127,700. Values are real (inflation-adjusted).*

---

### Scenario B — Exhaustive Rolling Windows (798 windows × 40 years)

> **“God Win Rate”** = % of windows where God's final portfolio > DCA's final portfolio.

| Strategy | God Win Rate | Median Adv | Mean Adv | Std Dev | Best | Worst |
|----------|-------------|-----------|---------|---------|------|-------|
| **Baseline (0% cash)** | **31%** (247/798) | −1.7% | −0.4% | 8.3% | +22.2% | −17.3% |
| **Maggiulli (real T-bill)** | **62%** (498/798) | +2.7% | +3.2% | 9.2% | +31.0% | −11.4% |
| **5-Year (real T-bill)** | **70%** (558/798) | +6.1% | +9.9% | 16.9% | +64.1% | −13.2% |
| **10-Year (real T-bill)** | **47%** (374/798) | −1.9% | +3.8% | 20.8% | +53.6% | −32.3% |

---

### Key Findings

**Baseline validates the replication:** God wins 31% of 40-year windows with 0% cash yield
on real total returns — matching Maggiulli's published ~30% figure.

**Real rates keep cash and equities in the same unit:** God's idle cash earns the
Fisher-equation real T-bill rate, keeping the cash yield in the same unit of account as the
Shiller Real Total Return index. During high-inflation periods God's war chest loses real
purchasing power; during deflation it grows. The cash yield assumption directly affects how
much God deploys at each bottom.

**Scenario A: Great Depression deflation creates a compounding bonus:** Strongly positive
real rates in 1929–1933 (deflation made every dollar grow in real terms) built a large war
chest deployed at the 1932 bottom. This deflation windfall more than offsets the
stagflation-era penalty over the 106-year full run: God Mag finishes +12.9% ahead of DCA;
God 10-Year +46.9%.

**God 10-Year wins only 47% of rolling windows — less than a coin flip:**
When each rolling window uses periods anchored to that window's start year
(not Gregorian-snapped), a decade of negative real rates is
devastating. The median 10-Year outcome is **−1.9%** (DCA wins on the median window).
The worst case is −32.3%. This is consistent with the economic reality: ten years of
stagflation-era real rates (e.g., 1972–1982) genuinely destroy the real value of
accumulated cash before God gets to deploy it.

**God 5-Year wins 70% of rolling windows — the superior window-relative period strategy:**
A 5-year accumulation window limits the damage from negative real-rate periods. Even
the worst 5-Year window only loses −13.2% to DCA. Half the downside exposure of the
10-Year strategy, but still a strong win rate.

**Why 10-Year (47%) trails 5-Year (70%):**
The 10-Year strategy suffers from two compounding forces during stagflation windows:
(1) a full decade of negative real rates erodes the war chest, and (2) the decade low
occurs at a less extreme discount than the 5-year low (more time for prices to recover
within the period). Shorter accumulation windows limit both effects.

**Data note:** All values are real (inflation-adjusted) using Shiller's Real Total Return
index. CPI sourced from Shiller's ie_data.xls throughout — single consistent source.
Real T-bill rate = Fisher equation: (1 + nominal annual effective) / (1 + trailing 12-month CPI) − 1.
FRED DGS1 (BEY) is converted to annual effective inside `load_bond_yields()` before
the Fisher equation is applied; Shiller's pre-1962 annual rate is passed through as-is.